In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Input
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [ ]:
print("Loading preprocessed dataset...")
# Make sure to use the absolute paths for Colab
df = pd.read_csv('/content/project/outputs/processed_features.csv')
feature_cols = [col for col in df.columns if col not in ['timestamp', 'target']]
X_raw = df[feature_cols].values
y_raw = df['target'].values

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
joblib.dump(scaler, '/content/project/models/data_scaler.pkl')

In [ ]:
def build_sequences(features, targets, sequence_length):
    X_seq, y_seq = [], []
    for i in range(len(features) - sequence_length):
        X_seq.append(features[i:(i + sequence_length)])
        y_seq.append(targets[i + sequence_length])
    return np.array(X_seq), np.array(y_seq)

In [ ]:
X_seq, y_seq = build_sequences(X_scaled, y_raw, 10)
train_size = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:train_size], X_seq[train_size:]
y_train, y_test = y_seq[:train_size], y_seq[train_size:]

In [ ]:
print("Initializing Optimized LSTM architecture...")
model = Sequential([
    Input(shape=(10, X_train.shape[2])),
    LSTM(32, activation='tanh', return_sequences=True),
    Dropout(0.3),
    LSTM(16, activation='tanh'),
    Dropout(0.3),
    Dense(8, activation='relu'),
    Dense(1)
])

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
              loss='mse',
              metrics=['mae'])

print("Setting up Callbacks (Checkpoint + Early Stopping)...")
# Silently saves the best weights
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='/content/project/models/dl_forecast_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [ ]:
# Automatically stops training if it overfits for 5 epochs in a row
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
print("Commencing model training loop...")
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=128,
    validation_split=0.15,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)

In [ ]:
print("Evaluating final metrics...")
# To evaluate the BEST model, we need to load the saved weights back in
model.load_weights('/content/project/models/dl_forecast_model.keras')
y_pred = model.predict(X_test)

print("--- FINAL BEST MODEL METRICS ---")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("Model saved perfectly to /content/project/models/dl_forecast_model.keras")